# SIED-Thai Original — Frozen WangchanBERTa + Optuna + 5 Runs

เวอร์ชันนี้ใช้ **SIED-Thai original / unbalanced** และยึดแนวทาง Strong Optimized ที่คุยกันไว้:

- Encoder: `airesearch/wangchanberta-base-att-spm-uncased`
- **Frozen encoder**
- X-Transformers decoder
- Optuna search **1 ครั้ง**
- ล็อก `best_params`
- Final training ใหม่ **5 seeds**
- Dataset split คงที่ด้วย `SPLIT_SEED = 42`
- Threshold tuning จาก validation set แยกในแต่ละ run
- Test set ชุดเดียวกันทุก run
- สรุป Mean / Median / Sample Variance / Sample SD

เพื่อประหยัดเวลา notebook จะ **cache hidden states จาก frozen encoder เพียงครั้งเดียว** แล้ว Optuna และ 5 final runs จะ train เฉพาะ decoder + pooling + classifier ซึ่งเทียบเท่ากับ frozen-encoder pipeline เมื่อ encoder อยู่ใน `eval()` mode.

In [ ]:
!pip install -q transformers==4.46.3 tokenizers==0.20.3 sentencepiece x-transformers optuna scikit-learn

In [ ]:
import os
import gc
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from x_transformers import Decoder

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    hamming_loss,
    classification_report
)

import optuna

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

In [ ]:
# =========================================================
# CONFIG
# =========================================================

MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"

MAX_LEN = 128
EPOCHS = 100
BATCH_SIZE = 64

NUM_LABELS = 6

# Optuna ทำเพียงครั้งเดียว
N_TRIALS = 25
OPTUNA_EPOCHS = 35

# Split ชุดเดียวกันทุก run
SPLIT_SEED = 42

# 5 independent training runs
TRAIN_SEEDS = [13, 21, 42, 87, 123]

LABEL_COLUMNS = [
    "Happiness",
    "Sadness",
    "Anger",
    "Disgust",
    "Surprise",
    "Fear"
]

TEXT_COLUMN = "Tweets"

# ถ้า deadline บีบมาก สามารถลด N_TRIALS เป็น 15 ได้
# แต่ final 5 runs ยังคงเหมือนเดิม

CANDIDATE_PATHS = [
    "/kaggle/input/datasets/kmkimmy/sied-thai/SIED-Thai.csv",
    "/content/SIED-Thai.csv",
    "/mnt/data/SIED-Thai.csv",
]

CSV_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)

if CSV_PATH is None:
    raise FileNotFoundError(
        "ไม่พบ SIED-Thai.csv กรุณา upload SIED-Thai original/unbalanced"
    )

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("DATASET:", CSV_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# =========================================================
# REPRODUCIBILITY
# =========================================================

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SPLIT_SEED)

In [ ]:
# =========================================================
# LOAD ORIGINAL / UNBALANCED SIED-THAI
# =========================================================

df = pd.read_csv(CSV_PATH)

required_columns = [TEXT_COLUMN] + LABEL_COLUMNS

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"dataset ไม่มี column: {col}")

df = df[required_columns].copy()

df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()

for col in LABEL_COLUMNS:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    ).fillna(0).astype(int)

df = df.dropna().reset_index(drop=True)

print("TOTAL ROWS:", len(df))

print("\nLABEL DISTRIBUTION")
print(df[LABEL_COLUMNS].sum())

print("\nLABEL CARDINALITY")
print(
    df[LABEL_COLUMNS]
    .sum(axis=1)
    .value_counts()
    .sort_index()
)

In [ ]:
# =========================================================
# FIXED TRAIN / VALIDATION / TEST SPLIT
# =========================================================
# Split คงเดิมทุก 5 runs เพื่อให้ variance สะท้อน training randomness

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SPLIT_SEED,
    shuffle=True
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SPLIT_SEED,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("TRAIN:", len(train_df))
print("VALIDATION:", len(val_df))
print("TEST:", len(test_df))

for name, frame in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df)
]:
    print(f"\n{name} LABEL DISTRIBUTION")
    print(frame[LABEL_COLUMNS].sum())

In [ ]:
# =========================================================
# TOKENIZER + FROZEN ENCODER
# =========================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

frozen_encoder = AutoModel.from_pretrained(
    MODEL_NAME
).to(DEVICE)

for param in frozen_encoder.parameters():
    param.requires_grad = False

frozen_encoder.eval()

print(
    "Trainable encoder parameters:",
    sum(p.numel() for p in frozen_encoder.parameters() if p.requires_grad)
)

In [ ]:
# =========================================================
# CACHE FROZEN ENCODER OUTPUTS ONCE
# =========================================================
# ทำให้ Optuna + 5 runs เร็วขึ้นมาก
# encoder ถูก freeze และ eval จึงให้ hidden states คงที่

@torch.no_grad()
def encode_frame(frame, batch_size=32):

    all_hidden = []
    all_masks = []
    all_labels = []

    texts = frame[TEXT_COLUMN].tolist()
    targets = frame[LABEL_COLUMNS].values.astype(np.float32)

    for start in range(0, len(texts), batch_size):

        batch_texts = texts[start:start + batch_size]

        enc = tokenizer(
            batch_texts,
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        input_ids = enc["input_ids"].to(DEVICE)
        attention_mask = enc["attention_mask"].to(DEVICE)

        hidden = frozen_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state

        # เก็บเป็น float16 เพื่อลด RAM
        all_hidden.append(
            hidden.cpu().to(torch.float16)
        )

        all_masks.append(
            attention_mask.cpu().to(torch.bool)
        )

        all_labels.append(
            torch.tensor(
                targets[start:start + batch_size],
                dtype=torch.float32
            )
        )

    return (
        torch.cat(all_hidden, dim=0),
        torch.cat(all_masks, dim=0),
        torch.cat(all_labels, dim=0)
    )


print("Encoding TRAIN...")
train_hidden, train_masks, train_labels = encode_frame(train_df)

print("Encoding VALIDATION...")
val_hidden, val_masks, val_labels = encode_frame(val_df)

print("Encoding TEST...")
test_hidden, test_masks, test_labels = encode_frame(test_df)

print("TRAIN HIDDEN:", train_hidden.shape)
print("VAL HIDDEN:", val_hidden.shape)
print("TEST HIDDEN:", test_hidden.shape)

# หลัง cache แล้วไม่ต้องเก็บ encoder บน GPU
del frozen_encoder
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# =========================================================
# CACHED FEATURE DATASET
# =========================================================

class CachedFeatureDataset(Dataset):

    def __init__(self, hidden, masks, labels):
        self.hidden = hidden
        self.masks = masks
        self.labels = labels

    def __len__(self):
        return len(self.hidden)

    def __getitem__(self, idx):
        return {
            "hidden": self.hidden[idx],
            "attention_mask": self.masks[idx],
            "labels": self.labels[idx]
        }


train_dataset = CachedFeatureDataset(
    train_hidden,
    train_masks,
    train_labels
)

val_dataset = CachedFeatureDataset(
    val_hidden,
    val_masks,
    val_labels
)

test_dataset = CachedFeatureDataset(
    test_hidden,
    test_masks,
    test_labels
)


def make_loaders(seed):

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator
    )

    train_eval_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    return (
        train_loader,
        train_eval_loader,
        val_loader,
        test_loader
    )

In [ ]:
# =========================================================
# POS WEIGHT
# =========================================================

label_counts = train_df[LABEL_COLUMNS].sum().values.astype(np.float32)
total_samples = len(train_df)
neg_counts = total_samples - label_counts

pos_weights_raw = neg_counts / (label_counts + 1e-6)

pos_weights_raw = torch.tensor(
    pos_weights_raw,
    dtype=torch.float32
).to(DEVICE)


def build_pos_weight(pos_weight_scale):

    scale = float(pos_weight_scale)

    weights = (
        1.0
        + (pos_weights_raw - 1.0) * scale
    )

    return weights.to(DEVICE)


print("LABEL COUNTS:", label_counts)
print("RAW POS WEIGHTS:", pos_weights_raw)

In [ ]:
# =========================================================
# DECODER MODEL
# =========================================================

class AttentionPooling(nn.Module):

    def __init__(self, hidden_size):
        super().__init__()

        self.attention = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x, attention_mask):

        scores = self.attention(x).squeeze(-1)

        scores = scores.masked_fill(
            ~attention_mask.bool(),
            -1e9
        )

        weights = torch.softmax(
            scores,
            dim=1
        )

        pooled = torch.sum(
            x * weights.unsqueeze(-1),
            dim=1
        )

        return pooled


class EmotionModel(nn.Module):

    def __init__(
        self,
        hidden_size=768,
        depth=1,
        heads=2,
        attn_dropout=0.15,
        ff_dropout=0.15,
        ff_mult=2
    ):

        super().__init__()

        self.decoder = Decoder(
            dim=hidden_size,
            depth=depth,
            heads=heads,
            attn_dropout=attn_dropout,
            ff_dropout=ff_dropout,
            ff_mult=ff_mult
        )

        self.pooling = AttentionPooling(
            hidden_size
        )

        self.dropout = nn.Dropout(0.30)

        self.fc = nn.Linear(
            hidden_size,
            NUM_LABELS
        )

    def forward(
        self,
        hidden,
        attention_mask
    ):

        # cached encoder outputs ถูกเก็บเป็น float16
        # cast ตาม dtype ของ decoder parameter
        dtype = next(self.parameters()).dtype
        x = hidden.to(
            DEVICE,
            dtype=dtype
        )

        mask = attention_mask.to(
            DEVICE
        )

        x = self.decoder(
            x,
            mask=mask.bool()
        )

        x = self.pooling(
            x,
            mask
        )

        x = self.dropout(x)

        return self.fc(x)

In [ ]:
# =========================================================
# EVALUATION + THRESHOLD TUNING
# =========================================================

def optimize_thresholds(
    y_true,
    y_probs,
    min_thr=0.20,
    max_thr=0.80,
    step=0.005
):

    candidates = np.arange(
        min_thr,
        max_thr + 1e-9,
        step
    )

    best_thresholds = []

    for i in range(NUM_LABELS):

        best_thr = 0.5
        best_f1 = -1.0
        best_precision = -1.0

        for thr in candidates:

            pred = (
                y_probs[:, i] >= thr
            ).astype(int)

            f1 = f1_score(
                y_true[:, i],
                pred,
                zero_division=0
            )

            precision = precision_score(
                y_true[:, i],
                pred,
                zero_division=0
            )

            if (
                f1 > best_f1
                or (
                    np.isclose(f1, best_f1)
                    and precision > best_precision
                )
            ):
                best_f1 = f1
                best_precision = precision
                best_thr = thr

        best_thresholds.append(
            best_thr
        )

    return np.array(best_thresholds)


@torch.no_grad()
def evaluate(
    model,
    loader,
    thresholds=None
):

    model.eval()

    all_probs = []
    all_labels = []

    for batch in loader:

        logits = model(
            batch["hidden"],
            batch["attention_mask"]
        )

        probs = torch.sigmoid(
            logits
        )

        all_probs.append(
            probs.cpu().numpy()
        )

        all_labels.append(
            batch["labels"].numpy()
        )

    y_probs = np.vstack(all_probs)
    y_true = np.vstack(all_labels)

    if thresholds is None:
        thresholds = np.array(
            [0.5] * NUM_LABELS
        )

    y_pred = (
        y_probs >= thresholds
    ).astype(int)

    per_label_f1 = f1_score(
        y_true,
        y_pred,
        average=None,
        zero_division=0
    )

    result = {
        "micro_precision": precision_score(
            y_true, y_pred,
            average="micro",
            zero_division=0
        ),
        "micro_recall": recall_score(
            y_true, y_pred,
            average="micro",
            zero_division=0
        ),
        "micro_f1": f1_score(
            y_true, y_pred,
            average="micro",
            zero_division=0
        ),
        "macro_precision": precision_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),
        "macro_recall": recall_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),
        "macro_f1": f1_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true, y_pred,
            average="weighted",
            zero_division=0
        ),
        "subset_accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "hamming_loss": hamming_loss(
            y_true,
            y_pred
        ),
        "per_label_precision": precision_score(
            y_true, y_pred,
            average=None,
            zero_division=0
        ),
        "per_label_recall": recall_score(
            y_true, y_pred,
            average=None,
            zero_division=0
        ),
        "per_label_f1": per_label_f1,
        "min_per_label_f1": float(
            np.min(per_label_f1)
        ),
        "labels": y_true,
        "preds": y_pred,
        "probs": y_probs
    }

    return result


def objective_score(result):

    return (
        0.45 * result["macro_f1"]
        + 0.25 * result["micro_f1"]
        + 0.20 * result["min_per_label_f1"]
        + 0.10 * result["subset_accuracy"]
    )

In [ ]:
# =========================================================
# TRAIN DECODER
# =========================================================

def train_model(
    model,
    train_loader,
    val_loader,
    lr,
    weight_decay,
    epochs,
    pos_weight_scale=0.0,
    label_smoothing=0.0,
    loss_name="bce",
    focal_gamma=2.0,
    verbose=False
):

    pos_weight = build_pos_weight(
        pos_weight_scale
    )

    bce_loss = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight,
        reduction="none"
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    total_steps = (
        len(train_loader)
        * epochs
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(
            0.06 * total_steps
        ),
        num_training_steps=total_steps
    )

    use_amp = DEVICE.type == "cuda"

    scaler = torch.cuda.amp.GradScaler(
        enabled=use_amp
    )

    for epoch in range(epochs):

        model.train()
        total_loss = 0.0

        for batch in train_loader:

            labels = batch["labels"].to(
                DEVICE
            )

            if label_smoothing > 0:
                labels_for_loss = (
                    labels * (1.0 - label_smoothing)
                    + 0.5 * label_smoothing
                )
            else:
                labels_for_loss = labels

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(
                enabled=use_amp
            ):

                logits = model(
                    batch["hidden"],
                    batch["attention_mask"]
                )

                loss_matrix = bce_loss(
                    logits,
                    labels_for_loss
                )

                if loss_name == "focal":

                    probs = torch.sigmoid(
                        logits
                    )

                    pt = (
                        probs * labels
                        + (1 - probs) * (1 - labels)
                    )

                    focal_weight = (
                        1 - pt
                    ).pow(focal_gamma)

                    loss = (
                        focal_weight
                        * loss_matrix
                    ).mean()

                else:
                    loss = loss_matrix.mean()

            scaler.scale(loss).backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            scheduler.step()

            total_loss += loss.item()

        if verbose and (
            epoch == 0
            or (epoch + 1) % 10 == 0
            or epoch == epochs - 1
        ):

            raw = evaluate(
                model,
                val_loader
            )

            thrs = optimize_thresholds(
                raw["labels"],
                raw["probs"]
            )

            tuned = evaluate(
                model,
                val_loader,
                thrs
            )

            print(
                f"Epoch {epoch+1:3d}/{epochs} | "
                f"loss={total_loss:.4f} | "
                f"val_micro={tuned['micro_f1']:.4f} | "
                f"val_macro={tuned['macro_f1']:.4f} | "
                f"val_acc={tuned['subset_accuracy']:.4f}"
            )

    return model

In [ ]:
# =========================================================
# OPTUNA — RUN ONCE
# =========================================================

OPTUNA_SEED = 42

optuna_train_loader, _, optuna_val_loader, _ = make_loaders(
    OPTUNA_SEED
)


def objective(trial):

    seed_everything(
        OPTUNA_SEED
    )

    depth = trial.suggest_int(
        "depth",
        1,
        3
    )

    heads = trial.suggest_categorical(
        "heads",
        [2, 4, 8]
    )

    attn_dropout = trial.suggest_float(
        "attn_dropout",
        0.05,
        0.30
    )

    ff_dropout = trial.suggest_float(
        "ff_dropout",
        0.05,
        0.30
    )

    ff_mult = trial.suggest_int(
        "ff_mult",
        2,
        4
    )

    lr = trial.suggest_float(
        "lr",
        1e-5,
        1e-4,
        log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-6,
        5e-3,
        log=True
    )

    pos_weight_scale = trial.suggest_float(
        "pos_weight_scale",
        0.0,
        0.60
    )

    label_smoothing = trial.suggest_float(
        "label_smoothing",
        0.0,
        0.04
    )

    loss_name = trial.suggest_categorical(
        "loss_name",
        ["bce", "focal"]
    )

    focal_gamma = trial.suggest_float(
        "focal_gamma",
        1.0,
        2.5
    )

    model = EmotionModel(
        hidden_size=768,
        depth=depth,
        heads=heads,
        attn_dropout=attn_dropout,
        ff_dropout=ff_dropout,
        ff_mult=ff_mult
    ).to(DEVICE)

    model = train_model(
        model,
        optuna_train_loader,
        optuna_val_loader,
        lr=lr,
        weight_decay=weight_decay,
        epochs=OPTUNA_EPOCHS,
        pos_weight_scale=pos_weight_scale,
        label_smoothing=label_smoothing,
        loss_name=loss_name,
        focal_gamma=focal_gamma,
        verbose=False
    )

    raw = evaluate(
        model,
        optuna_val_loader
    )

    thresholds = optimize_thresholds(
        raw["labels"],
        raw["probs"]
    )

    result = evaluate(
        model,
        optuna_val_loader,
        thresholds
    )

    score = objective_score(
        result
    )

    trial.set_user_attr(
        "val_macro_f1",
        float(result["macro_f1"])
    )

    trial.set_user_attr(
        "val_micro_f1",
        float(result["micro_f1"])
    )

    trial.set_user_attr(
        "val_min_per_label_f1",
        float(result["min_per_label_f1"])
    )

    trial.set_user_attr(
        "val_subset_accuracy",
        float(result["subset_accuracy"])
    )

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return score


sampler = optuna.samplers.TPESampler(
    seed=OPTUNA_SEED,
    multivariate=True
)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler
)

# baseline configuration as first trial
study.enqueue_trial({
    "depth": 1,
    "heads": 2,
    "attn_dropout": 0.20,
    "ff_dropout": 0.20,
    "ff_mult": 2,
    "lr": 2e-5,
    "weight_decay": 1e-3,
    "pos_weight_scale": 0.0,
    "label_smoothing": 0.0,
    "loss_name": "bce",
    "focal_gamma": 2.0
})

study.optimize(
    objective,
    n_trials=N_TRIALS
)

best_params = study.best_params

print("\nBEST OPTUNA SCORE:", study.best_value)
print("\nBEST PARAMS")
print(best_params)

best_params_path = os.path.join(
    OUTPUT_DIR,
    "sied_original_optuna_best_params.json"
)

with open(
    best_params_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        best_params,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:", best_params_path)

In [ ]:
# =========================================================
# FINAL 5 INDEPENDENT RUNS
# =========================================================

all_run_rows = []
all_per_label_rows = []

for run_idx, seed in enumerate(
    TRAIN_SEEDS,
    start=1
):

    print("\n" + "=" * 90)
    print(
        f"FINAL RUN {run_idx}/5 | seed={seed}"
    )
    print("=" * 90)

    seed_everything(
        seed
    )

    (
        train_loader,
        train_eval_loader,
        val_loader,
        test_loader
    ) = make_loaders(seed)

    model = EmotionModel(
        hidden_size=768,
        depth=best_params["depth"],
        heads=best_params["heads"],
        attn_dropout=best_params["attn_dropout"],
        ff_dropout=best_params["ff_dropout"],
        ff_mult=best_params["ff_mult"]
    ).to(DEVICE)

    model = train_model(
        model,
        train_loader,
        val_loader,
        lr=best_params["lr"],
        weight_decay=best_params["weight_decay"],
        epochs=EPOCHS,
        pos_weight_scale=best_params["pos_weight_scale"],
        label_smoothing=best_params["label_smoothing"],
        loss_name=best_params["loss_name"],
        focal_gamma=best_params["focal_gamma"],
        verbose=True
    )

    # tune thresholds on validation for this run only
    val_raw = evaluate(
        model,
        val_loader
    )

    best_thresholds = optimize_thresholds(
        val_raw["labels"],
        val_raw["probs"]
    )

    test_result = evaluate(
        model,
        test_loader,
        best_thresholds
    )

    row = {
        "run": run_idx,
        "seed": seed,
        "subset_accuracy": test_result["subset_accuracy"],
        "micro_precision": test_result["micro_precision"],
        "micro_recall": test_result["micro_recall"],
        "micro_f1": test_result["micro_f1"],
        "macro_precision": test_result["macro_precision"],
        "macro_recall": test_result["macro_recall"],
        "macro_f1": test_result["macro_f1"],
        "weighted_f1": test_result["weighted_f1"],
        "hamming_loss": test_result["hamming_loss"]
    }

    all_run_rows.append(
        row
    )

    print("\nTEST RESULT")
    for key, value in row.items():
        if key not in ["run", "seed"]:
            print(
                f"{key:20s}: {value:.6f}"
            )

    print("\nTHRESHOLDS")
    for label, thr in zip(
        LABEL_COLUMNS,
        best_thresholds
    ):
        print(
            f"{label:12s}: {thr:.3f}"
        )

    for i, label in enumerate(
        LABEL_COLUMNS
    ):

        all_per_label_rows.append({
            "run": run_idx,
            "seed": seed,
            "label": label,
            "precision": test_result["per_label_precision"][i],
            "recall": test_result["per_label_recall"][i],
            "f1": test_result["per_label_f1"][i]
        })

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


results_df = pd.DataFrame(
    all_run_rows
)

per_label_df = pd.DataFrame(
    all_per_label_rows
)

print("\nALL 5 RUNS")
display(results_df)

In [ ]:
# =========================================================
# MEAN / MEDIAN / VARIANCE / STANDARD DEVIATION
# =========================================================

metric_columns = [
    "subset_accuracy",
    "micro_precision",
    "micro_recall",
    "micro_f1",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_f1",
    "hamming_loss"
]

summary_rows = []

for metric in metric_columns:

    values = results_df[
        metric
    ].astype(float)

    summary_rows.append({
        "metric": metric,
        "mean": values.mean(),
        "median": values.median(),
        # sample variance / SD
        "variance": values.var(ddof=1),
        "std": values.std(ddof=1),
        "min": values.min(),
        "max": values.max()
    })


summary_df = pd.DataFrame(
    summary_rows
)

print("\n5-RUN STATISTICAL SUMMARY")
display(summary_df)

In [ ]:
# =========================================================
# PER-LABEL SUMMARY ACROSS 5 RUNS
# =========================================================

per_label_summary = (
    per_label_df
    .groupby("label")
    .agg(
        precision_mean=("precision", "mean"),
        precision_median=("precision", "median"),
        precision_variance=("precision", lambda x: x.var(ddof=1)),
        precision_std=("precision", lambda x: x.std(ddof=1)),

        recall_mean=("recall", "mean"),
        recall_median=("recall", "median"),
        recall_variance=("recall", lambda x: x.var(ddof=1)),
        recall_std=("recall", lambda x: x.std(ddof=1)),

        f1_mean=("f1", "mean"),
        f1_median=("f1", "median"),
        f1_variance=("f1", lambda x: x.var(ddof=1)),
        f1_std=("f1", lambda x: x.std(ddof=1))
    )
    .reset_index()
)

print("\nPER-LABEL 5-RUN SUMMARY")
display(per_label_summary)

In [ ]:
# =========================================================
# EXPORT
# =========================================================

paths = {
    "raw_results": os.path.join(
        OUTPUT_DIR,
        "sied_original_frozen_optuna_5runs_raw.csv"
    ),
    "summary": os.path.join(
        OUTPUT_DIR,
        "sied_original_frozen_optuna_5runs_summary.csv"
    ),
    "per_label_raw": os.path.join(
        OUTPUT_DIR,
        "sied_original_frozen_optuna_5runs_per_label_raw.csv"
    ),
    "per_label_summary": os.path.join(
        OUTPUT_DIR,
        "sied_original_frozen_optuna_5runs_per_label_summary.csv"
    ),
    "optuna_trials": os.path.join(
        OUTPUT_DIR,
        "sied_original_optuna_trials.csv"
    )
}

results_df.to_csv(
    paths["raw_results"],
    index=False,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    paths["summary"],
    index=False,
    encoding="utf-8-sig"
)

per_label_df.to_csv(
    paths["per_label_raw"],
    index=False,
    encoding="utf-8-sig"
)

per_label_summary.to_csv(
    paths["per_label_summary"],
    index=False,
    encoding="utf-8-sig"
)

study.trials_dataframe().to_csv(
    paths["optuna_trials"],
    index=False,
    encoding="utf-8-sig"
)

print("EXPORTED FILES")
for name, path in paths.items():
    print(name, "->", path)